# 작은 diffusion LM으로 한국어 동화 만들기 — 80/10/10 마스킹

이 장에서는 직전 영어 챕터와 똑같은 4.29M짜리 작은 mask-diffusion 언어 모델로 한국어 동화를 생성합니다. 전부 `[MASK]`로 채운 빈 캔버스에서 출발해도, 학습을 마친 모델은 이런 이야기를 끝까지 이어 씁니다.

> 옛날 옛적에 팀이라는 작은 소년이 있었어요. 팀은 장난감 가지고 노는 것을 매우 좋아했지요…

> 공원에 도착해서 맥스는 맥스라는 큰 공을 봤어요. "와, 정말 멋진 공이야! 우리 공으로 같이 놀자!"…

인물과 대화, 배경과 서사가 모두 들어 있습니다. 고정 마스킹 비율 $t=0.15$에서 빈칸 복원 top-1 정확도는 0.652까지 올라가고(영어 Ch 33은 0.717), `train_loss`는 4.13까지 내려갑니다.

이 결과를 만든 핵심은 **BERT가 원래 쓰던 80/10/10 마스킹** 한 가지입니다. 마스킹 대상 자리를 전부 `[MASK]`로 바꾸지 않고, 80%만 `[MASK]`로 두고 10%는 랜덤 토큰, 10%는 원본 그대로 둡니다. 영어 Ch 33과 모델·데이터 규모는 같고, 한국어에서 결정적으로 달라지는 건 이 마스킹뿐입니다. 왜 이 한 가지가 한국어에서 그토록 결정적인지는, 실습으로 동작하는 결과를 먼저 본 뒤 **해부** 절에서 순진한 100% `[MASK]` 방식과 직접 대조하며 규명합니다.

## 📊 추적표 (Phase 5 — diffusion 라인)

| 챕터 | 언어 | 토크나이저 | 마스킹 방식 | 결과 |
|---|---|---|---|---|
| Ch 33 (성공) | 영어 (TinyStories) | ByteLevel BPE, vocab 2048 | 100% `[MASK]` | train_loss 낮고 생성 coherent, top-1 acc 0.717 |
| **Ch 34 (해결)** | **한국어 (TinyStories-Korean)** | **ByteLevel BPE, vocab 4000** | **80/10/10 (`[MASK]` / 랜덤 / 원본)** | **순진한 이식은 train_loss 7.06 · acc 0.081로 붕괴 → 80/10/10으로 train_loss 4.13 · acc 0.652 회복, 생성 coherent** |

Ch 33과 Ch 34의 모델 본체(`BertForMaskedLM`, hidden 256/4L/4H, ~4.29M)와 샘플러(carry-over semi-AR, block 32 + 반복억제)는 같습니다. 바뀐 것은 학습 언어와, 그 변화가 드러낸 마스킹 방식 하나입니다. 영어에서 100% `[MASK]`로도 버티던 레시피가 한국어에서 무너지고, 마스킹 대상의 일부를 랜덤 토큰과 원본으로 섞는 80/10/10을 적용하자 손실의 벽 7.0을 완전히 돌파합니다.

## 🔄 변경점 (Diff from Ch 33)

| 항목 | 이전 (Ch 33) | 이번 (Ch 34) | 왜 바꾸나 |
|---|---|---|---|
| 언어 | 영어 | 한국어 | Ch 33의 영어 성공 레시피가 한국어에서도 통하는지 확인하려는 것이 출발점입니다. 결과적으로 이 변화가 마스킹 방식의 숨은 결함을 끌어냈습니다. |
| 데이터 | TinyStories | TinyStories-Korean (`g0ster/TinyStories-Korean`, 50,000 stories → 75,941 chunks, 약 9.7M 토큰) | Ch 26 한국어 AR과 같은 코퍼스를 써서 "같은 데이터로 AR은 되는데 diffusion은 왜 안 되나"를 깔끔하게 대조하기 위해서입니다. |
| vocab | 2048 | 4000 | 한국어는 조사·어미가 붙는 교착어라 영어보다 토큰 다양성이 커서 vocab을 약간 키웁니다. Ch 26 한국어 AR도 4000을 썼습니다. |
| 마스킹 | 100% `[MASK]` | 80/10/10 (80% `[MASK]`, 10% 랜덤 토큰, 10% 원본 유지) | 한국어는 조사·어미의 유니그램 예측성이 높아, "[MASK]면 고빈도 토큰 찍기" 지름길의 유혹이 영어보다 훨씬 강합니다. 마스킹 대상에 랜덤·원본을 섞어 그 지름길을 막아야 모델이 문맥을 실제로 학습합니다. |

핵심 메시지는 이렇습니다. 우리는 언어 하나만 바꾼 줄 알았는데, 그 변화가 마스킹 방식의 결함을 드러냈습니다. 영어 TinyStories는 어휘와 구문이 단순하고 반복적이라 100% `[MASK]`만으로도 모델이 문맥을 배울 여지가 있었습니다. 한국어는 같은 자리를 조사·어미의 빈도만으로 그럴듯하게 채울 수 있어, 100% `[MASK]`는 모델에게 "주변을 안 봐도 되는" 지름길을 열어 줍니다. 그래서 변경점은 표면상 "언어" 하나지만, 실질적으로는 그 언어가 끌어낸 "마스킹 방식"까지 두 칸이 함께 움직인 장입니다.

## 📐 마스킹 노트 — 80/10/10이 한국어 diffusion의 정답

이번 장의 학습 코드는 마스킹 대상으로 뽑힌 자리를 전부 `[MASK]`로 바꾸지 않습니다. BERT가 원래 쓰던 비율 그대로 셋으로 나눕니다.

- **80% → `[MASK]`** — 일반적인 빈칸 복원
- **10% → 랜덤 토큰** — 엉뚱한 토큰으로 바꿔놓고 정답을 맞히게 함
- **10% → 원본 그대로 유지** — 입력은 멀쩡한데 그 자리에도 손실이 걸림

핵심은 두 소수파가 하는 일입니다. **10% 원본 유지**는 "정답을 맞혀야 하는 자리가 꼭 `[MASK]`인 것은 아니다"라고 알려줍니다. 모델은 `[MASK]` 표식에만 기대 답을 고를 수 없고, 멀쩡해 보이는 토큰도 문맥과 맞는지 매번 검증해야 합니다. **10% 랜덤 토큰**은 "눈에 보이는 입력이 틀렸을 수 있으니 교정하라"고 강제합니다. 입력을 그대로 믿는 대신 문맥과 대조하는 습관이 생깁니다. 두 장치가 합쳐지면 모델은 표식이 아니라 문맥을 읽게 됩니다.

### 수치 예시 — 128토큰 시퀀스에서 어떻게 쪼개지는가

마스킹 비율 $t$를 뽑으면 시퀀스 길이 $L=128$에 대해 약 $tL$개 자리가 마스킹 대상이 되고, 그 안에서 다시 80/10/10으로 나뉩니다.

| 마스킹 비율 $t$ | 마스킹 대상 ($\approx tL$) | `[MASK]` (80%) | 랜덤 (10%) | 원본 유지 (10%) |
|:---:|:---:|:---:|:---:|:---:|
| 0.15 | 20 | 16 | 2 | 2 |
| 0.30 | 38 | 31 | 4 | 4 |
| 0.50 | 64 | 51 | 6 | 6 |

(마스킹 대상 수와 80/10/10 분해는 반올림 때문에 합이 ±1 어긋날 수 있습니다. $t=0.15$ 행만 16+2+2=20으로 정확히 떨어집니다.)

표를 보면 $t=0.15$에서 정답을 맞혀야 하는 20개 자리 중 4개(랜덤 2 + 원본 2)는 `[MASK]` 표식이 아예 없습니다. 이 4개가 모델에게 "표식이 아니라 문맥을 보라"고 끊임없이 압박하는 장치입니다. 가변 마스킹 $t \sim U(0.05, 1)$은 직전 챕터 그대로 유지합니다.

손실은 이 마스킹 대상 자리들의 평균 교차 엔트로피로 계산합니다. 직전 챕터에서 쓰던 시간 가중 $1/t$는 기댓값 상 마스크 토큰당 평균 CE와 상수배로 같으므로, 여기서는 표준 MLM과 눈높이를 맞추기 위해 가중 없는 plain CE(`BertForMaskedLM` 기본)를 그대로 씁니다.

### 순진한 100% `[MASK]`는 지름길을 연다

같은 모델·데이터에서 마스킹 대상을 전부 `[MASK]`로만 바꾸면, 모델은 두 가지를 알아챕니다. 정답을 맞혀야 하는 자리는 **항상** `[MASK]` 자리이고, `[MASK]` 자체엔 의미가 없으니 주변만 보면 된다는 것입니다. 한국어는 조사("-가", "-를", "-에")와 어미("-요", "-어요", "-습니다")의 빈도가 압도적으로 높아, 문맥을 거의 안 보고 고빈도 토큰만 찍어도 손실이 빠르게 떨어집니다. 그래서 모델은 "`[MASK]`면 흔한 토큰 찍기"라는 유니그램 지름길에 멈춥니다. 균등하게 찍을 때의 손실이 $\ln 4000 \approx 8.29$인데, 이 지름길은 빈도 정보만큼만 내려간 자리에서 정체합니다.

80/10/10의 10% 원본 유지(문맥 검증 강제)와 10% 랜덤(교정 강제)이 바로 이 지름길을 막습니다. 그래서 한국어 diffusion에서는 이 마스킹이 선택이 아니라 정답입니다.

이 한 가지 수정으로 `train_loss`가 4.13까지 내려가고 고정-$t$(0.15) top-1 정확도가 0.081에서 0.652로 뜁니다. 곧 실습에서 직접 확인합니다.

## 🔤 토크나이저 노트 — 한국어용 ByteLevel BPE (vocab 4000)

이 챕터는 ByteLevel BPE 토크나이저를 직접 학습해서 씁니다. 어휘 크기는 4000으로, 영어 직전 챕터의 2048보다 키웠습니다. 한국어는 조사와 어미가 어간에 붙는 교착어라 같은 어근도 표면형이 다양하게 늘어나기 때문에, 어휘가 너무 작으면 토큰이 잘게 부서져 한 문장의 토큰 수가 불필요하게 길어집니다. 한국어 자기회귀 모델을 다룬 챕터에서도 같은 4000을 썼습니다.

ByteLevel 방식은 텍스트를 일단 바이트 단위로 본 뒤 그 위에서 BPE 병합을 학습합니다. 학습 시 `initial_alphabet`으로 256개 바이트를 전부 포함시켜 두기 때문에, 어떤 한글 음절이나 처음 보는 문자가 들어와도 최소한 바이트 단위로는 항상 표현할 수 있습니다. 덕분에 `[UNK]`로 빠지는 경우가 사실상 사라집니다. 여기에 마스크 학습과 패딩에 필요한 특수 토큰 `[PAD]`, `[UNK]`, `[MASK]`를 더했습니다. `[MASK]`는 이번 챕터의 80/10/10 마스킹에서 핵심 역할을 합니다.

예를 들어 "옛날 옛날에 작은 토끼가 숲으로 갔어요"라는 문장을 토큰화하면, "옛날", "작은"처럼 자주 등장하는 덩어리는 하나의 토큰으로 묶이고, 드문 글자나 조사 경계는 더 작은 서브워드 또는 바이트 조각으로 쪼개집니다. 같은 어근이라도 "갔어요", "갔다"처럼 어미가 달라지면 뒤쪽 토큰이 갈라지는데, 어휘를 4000까지 확보한 덕에 흔한 어미 패턴은 통째로 한 토큰으로 유지되어 시퀀스가 지나치게 길어지지 않습니다.

## 🚀 실습 — 직접 학습하고 생성해 보기

80/10/10 콜레이터로 작은 모델을 학습시켜 한국어 동화를 생성해 봅니다. (80/10/10이 무엇인지는 위 마스킹 노트에서 봤고, 순진한 100% `[MASK]`가 왜 무너지는지는 뒤 🔬 해부에서 다룹니다.) T4에서 30000 step, 약 20분.

In [1]:
%pip install -q -U transformers tokenizers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 22.4/48.9 MB 219.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 48.9/48.9 MB 171.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 48.9/48.9 MB 171.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.2 MB/s eta 0:00:00


In [2]:
import math, time, torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
SEED=42; torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()
print("device", device, "| fp16", USE_FP16)

device cuda | fp16 True


### 1. 한국어 TinyStories 복원 (Ch 26과 동일)

In [3]:
EOT="<|endoftext|>"; N_TRAIN,N_VAL,MAXL=50_000,500,1_500_000
def rebuild(split,n,maxl):
    stories,buf=[],[]
    for i,ex in enumerate(load_dataset("g0ster/TinyStories-Korean",split=split,streaming=True)):
        if i>=maxl or len(stories)>=n: break
        line=(ex["text"] or "").strip()
        if line==EOT:
            s=" ".join(buf).strip()
            if s: stories.append(s)
            buf=[]
        elif line: buf.append(line)
    if buf and len(stories)<n:
        s=" ".join(buf).strip()
        if s: stories.append(s)
    return stories[:n]
raw_train=Dataset.from_dict({"text":rebuild("train",N_TRAIN,MAXL)})
raw_val=Dataset.from_dict({"text":rebuild("validation",N_VAL,50_000)})
print("stories", len(raw_train), len(raw_val))
print(raw_train[0]["text"][:120])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/789 [00:00<?, ?B/s]

stories 50000 500
한때 벤이라는 이름의 어린 소년이 있었어요. 벤은 주변 세계를 탐험하는 것을 좋아했답니다. 그는 가게에 전시되어 있던 아름다운 꽃병들 같은 멋진 것들을 많이 봤어요. 어느 날, 벤은 가게를 거닐다가 정말 특별한 꽃병


### 2. BPE 4000 + initial_alphabet + [MASK]

In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
VOCAB=4000
def corpus_iter(bs=1000):
    for i in range(0,len(raw_train),bs): yield raw_train[i:i+bs]["text"]
_tk=Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)
_tk.decoder=decoders.ByteLevel()
_tk.train_from_iterator(corpus_iter(), trainer=trainers.BpeTrainer(
    vocab_size=VOCAB, special_tokens=["[PAD]","[UNK]","[MASK]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
tokenizer=PreTrainedTokenizerFast(tokenizer_object=_tk, pad_token="[PAD]", unk_token="[UNK]", mask_token="[MASK]")
print("vocab", tokenizer.vocab_size, "mask_id", tokenizer.mask_token_id)

vocab 4000 mask_id 2


### 3. 토큰화 + group

In [5]:
BLOCK=128
tt=raw_train.map(lambda b: tokenizer(b["text"],add_special_tokens=False), batched=True, remove_columns=raw_train.column_names)
tv=raw_val.map(lambda b: tokenizer(b["text"],add_special_tokens=False), batched=True, remove_columns=raw_val.column_names)
def group(b):
    cat=sum(b["input_ids"],[]); n=(len(cat)//BLOCK)*BLOCK
    return {"input_ids":[cat[i:i+BLOCK] for i in range(0,n,BLOCK)]}
lm_train=tt.map(group,batched=True,remove_columns=tt.column_names)
lm_val=tv.map(group,batched=True,remove_columns=tv.column_names)
print("chunks", len(lm_train), len(lm_val))

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

chunks 75941 746


### 4. ★수정 — diffusion 콜레이터에 80/10/10 (순진한 100% [MASK] 대신)

가변 마스킹률 t는 유지(생성용)하되, 선택된 자리에 80% [MASK] / 10% 랜덤 / 10% 원본유지. 이게 모델이 [MASK]→유니그램 지름길로 새는 걸 막는다.

In [6]:
N_SPECIAL=3
class DiffMLMCollator:
    def __init__(self, tok, eps=0.05, tmax=1.0, seed=SEED):
        self.mask_id=tok.mask_token_id; self.vocab=tok.vocab_size
        self.eps=eps; self.tmax=tmax; self.gen=torch.Generator().manual_seed(seed)
    def __call__(self, ex):
        ids=torch.tensor([e["input_ids"] for e in ex], dtype=torch.long)
        B,L=ids.shape
        t=torch.rand(B,generator=self.gen)*(self.tmax-self.eps)+self.eps
        sel=torch.rand(B,L,generator=self.gen)<t.unsqueeze(1)
        no=~sel.any(1)
        if no.any():
            j=torch.randint(0,L,(int(no.sum()),),generator=self.gen); sel[no,j]=True
        labels=ids.clone(); labels[~sel]=-100
        inp=ids.clone()
        r=torch.rand(B,L,generator=self.gen)
        inp[sel&(r<0.8)]=self.mask_id                                    # 80% [MASK]
        rp=sel&(r>=0.8)&(r<0.9); nr=int(rp.sum())                        # 10% 랜덤
        if nr: inp[rp]=torch.randint(N_SPECIAL,self.vocab,(nr,),generator=self.gen)
        # 10% 원본 유지
        return {"input_ids":inp,"attention_mask":torch.ones(B,L,dtype=torch.long),"labels":labels}
coll=DiffMLMCollator(tokenizer)

### 5. 작은 모델 (256/4L) — 용량 아닌 마스킹이 문제였음

In [7]:
from transformers import BertConfig, BertForMaskedLM
cfg=BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=256, num_hidden_layers=4,
               num_attention_heads=4, intermediate_size=1024,
               max_position_embeddings=BLOCK, pad_token_id=tokenizer.pad_token_id)
model=BertForMaskedLM(cfg).to(device)
print("params(M)", round(model.num_parameters()/1e6,2))

params(M) 4.29


### 6. 학습 — plain CE(BertForMaskedLM 기본) + lr 5e-4, 30000 step

In [8]:
from transformers import Trainer, TrainingArguments
args=TrainingArguments(output_dir="./out34", max_steps=30000,
    per_device_train_batch_size=64, learning_rate=5e-4, weight_decay=0.01,
    warmup_steps=1000, lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
    logging_steps=500, save_strategy="no", report_to="none", remove_unused_columns=False, seed=SEED)
trainer=Trainer(model=model, args=args, train_dataset=lm_train, data_collator=coll)
t0=time.time(); r=trainer.train()
print(f"elapsed {(time.time()-t0)/60:.2f}min | step {r.global_step} | train_loss {r.training_loss:.4f} | baseline ln(V) {math.log(tokenizer.vocab_size):.4f}")

Step,Training Loss
500,7.181071
1000,6.543790
1500,6.420795
2000,6.363970
2500,6.231786
3000,5.907821
3500,5.315859
4000,4.798746
4500,4.572670
5000,4.417235


elapsed 20.20min | step 30000 | train_loss 4.1260 | baseline ln(V) 8.2940


### 7. carry-over 샘플러로 한국어 생성

In [9]:
@torch.no_grad()
def generate(model, length=128, block=32, temperature=0.8, top_p=0.92, top_k=0,
             rep_penalty=1.3, no_immediate_repeat=True, prompt_ids=None):
    """carry-over semi-AR + 반복 억제(rep penalty / 인접중복 금지 / top-p)."""
    model.eval()
    mask_id = tokenizer.mask_token_id
    x = torch.full((1, length), mask_id, dtype=torch.long, device=device)
    fixed = torch.zeros(length, dtype=torch.bool, device=device)
    if prompt_ids is not None:
        p = torch.tensor(prompt_ids[:length], device=device)
        x[0, :len(p)] = p; fixed[:len(p)] = True
    nblocks = (length + block - 1) // block
    for b in range(nblocks):
        lo, hi = b * block, min((b + 1) * block, length)
        steps = hi - lo
        for s in range(steps):
            logits = model(input_ids=x).logits[0].float()        # (L, V)
            logits[:, mask_id] = -1e9
            # 반복 패널티: 이미 확정된 토큰들의 로짓을 깎음
            if rep_penalty and rep_penalty != 1.0:
                comm = x[0][x[0] != mask_id]
                if comm.numel() > 0:
                    u = torch.unique(comm)
                    col = logits[:, u]
                    logits[:, u] = torch.where(col > 0, col / rep_penalty, col * rep_penalty)
            # 인접중복 금지: 각 자리에서 '왼쪽 토큰과 같은 토큰' 예측 차단
            if no_immediate_repeat:
                left = torch.roll(x[0], 1); left[0] = mask_id
                valid = left != mask_id
                logits[valid, left[valid]] = -1e9
            probs = (logits / max(temperature, 1e-6)).softmax(-1)
            if top_k and top_k > 0:
                kth = probs.topk(top_k, dim=-1).values[:, -1, None]
                probs = probs.masked_fill(probs < kth, 0.0)
            if top_p and top_p < 1.0:
                sp, si = probs.sort(dim=-1, descending=True)
                rm = (sp.cumsum(-1) - sp) > top_p
                sp = sp.masked_fill(rm, 0.0)
                probs = torch.zeros_like(probs).scatter(-1, si, sp)
            probs = probs / probs.sum(-1, keepdim=True).clamp_min(1e-9)
            pred = torch.multinomial(probs, 1).squeeze(-1)
            conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
            cur = (x[0] == mask_id) & (~fixed)
            cur[:lo] = False; cur[hi:] = False
            nleft = int(cur.sum())
            if nleft == 0: break
            nreveal = nleft if s == steps - 1 else max(1, nleft // (steps - s))
            cc = conf.clone(); cc[~cur] = -1e9
            idx = cc.topk(nreveal).indices
            x[0, idx] = pred[idx]
    return tokenizer.decode(x[0], skip_special_tokens=True)

pid = tokenizer("옛날 옛날에", add_special_tokens=False)["input_ids"]
torch.manual_seed(SEED)
print("=== unconditional (all-[MASK] -> generate, default sampler) ===")
for i in range(3):
    print(f"[{i}] {generate(model)[:340]}")
print("\n=== conditional (prompt 'Once upon a time' fixed) ===")
for i in range(3):
    print(f"[{i}] {generate(model, prompt_ids=pid)[:340]}")

pid = tokenizer("옛날 옛날에", add_special_tokens=False)["input_ids"]
torch.manual_seed(SEED)
print("=== conditional ('옛날 옛날에') ===")
for i in range(3):
    print(f"[{i}] {generate(model, prompt_ids=pid)[:300]}")
print("\n=== unconditional ===")
for i in range(2):
    print(f"[{i}] {generate(model)[:300]}")

=== unconditional (all-[MASK] -> generate, default sampler) ===


[0]  공원에 도착해서 맥스는 맥스라는 큰 공을 봤어요. "와, 정말 멋진 공이야! 우리 공으로 같이 놀자!" 맥스는 웃으며 "그래, 가자!"라고 말했어요. 그들은 함께 놀면서 재미있게 놀았어요. 결국, 릴리와 그녀의 친구들은 다시 즐겁게 놀 수 있었어요. 맥스와 릴리는 빨간 공을 가지고 놀며 즐거운 시간을 보냈답니다.어느 날, 팀이라는 작은 소년이 놀러 갔어요. 그는 크고 빨간색의 개의 주인이 있었죠. 팀은 긴 키 긴 꼬리를 가진 갈색 개를 만났어요. 그 개는 팀과 릴리에게 그 개와 놀고 싶어 했어요. 그녀는 팀에게 다가가서 물었어요, "안녕, 개야! 나랑 같이 놀아도 될래?" 고양이는 팀을 쳐다보며 대답


[1] 옛날 옛적에, 팀이라는 작은 소년이 있었어요. 팀은 하루 종일 장난감 자동차를 가지고 노는 것을 좋아했죠. 맑은 어느 날, 팀은 마당에서 수 있는 가장 좋아하는 장난감을 발견했어요. 그는 그 자동차와 함께 놀고 싶어서 마당을 뛰어다니며 뛰어다녔죠. 팀은 기뻐하며 친구들에게 자신의 장난감 자동차로 공으로 놀기 시작했어요. 그들은 웃으며 놀면서 정말 재미있게 놀았어요. 그런데 뜻밖의 일이 벌어졌어요. 장난감이 일어났어요. 상자 안에는 진짜 자동차가 아니었어요. 바위가 아니라 큰 공이 움직이기 시작한 거예요! 팀과 친구들은 놀랐지만, 그게 바로 마법 같은 것 같아요!"라고 말했어요. 둘은 번갈아 가며 공으로


[2]  장난감 자동차를 가지고 노는 것을 좋아했습니다. 어느 날, 팀은 가장 좋아하는 자동차와 놀고 있었습니다. 그는 자신의 자동차들과 함께 차를 운전하고 싶어 했습니다. 팀은 매우 빠른적이라고 생각했습니다. 그래서 그는 친구 수에게 갔습니다. 그녀는 친구인 소년에게 같이 놀아달라고 부탁했습니다. 그들은 모두 그 장난감 자동차로 놀았습니다. 하지만 갑자기 팀의 자동차가 고장 나서 달아나버렸습니다. 팀과 수는 더 이상 슬프고 외롭지 않았습니다. 결국 팀의 엄마는 다시 장난감 자동차를 고칠 수 있게 되었습니다. 이제 둘 다 재미있게 놀았지만, 곧 손은 사라졌습니다. 그들은 자신들의 장난감 자동차를 질주할 수 있

=== conditional (prompt 'Once upon a time' fixed) ===


[0] 옛날 옛날에, 수라는 이름의 작은 소녀가 있었어요. 그녀는 장난감 자동차를 가지고 노는 것을 정말 좋아했지요. 어느 날, 수는 자신의 자동차에 큰 장난감 자동차를 발견했어요. 그 차는 아주 무거웠어요. 그래서 친구 수에게 그 자동차로 놀고 싶어 했어요. 수가 말했어요, "아니야! 이건 내 차야!" 하지만 팀은 듣지 않았어요. 그들은 팀의 말을 듣고 도와주러 왔어요. 그는 팀에게 달려가서 "팀, 이제 난 너랑 같이 놀 수 없어."라고 했죠. 수의 엄마가 그를 보고 말했죠, "팀아, 네 자동차가 고장 나갈 수 있어?" 팀은 슬퍼하며 엄마에게 물었어요, "아니, 팀이야, 우리 차를 고칠게 해. 너는 내 장난


...

[0] 옛날 옛날에, 큰 빨간 기차가 있었어요. 그 기차는 많은 색깔의 바퀴를 가지고 있었지요. 매일 파란 기차를 함께 노는 것을 좋아했답니다. 어느 날, 파란색 기차는 빨간색이고 노란색 기차를 발견했어요. 새도 그 차로를 고치고 싶어 했죠. 차는 매우 신나했어요! 새로운 친구인 작은 초록색 자동차와 "나 같이 놀자!"라고 말했죠. 그들은 모두 하루 종일 놀았어요. 해가 지기 시작하자, 파란 자동차가 점점 더 빠르게 달렸어요. 파란 공은 빨랐죠. 하늘은 높이 올라갔죠. 파란 차는 정말 행복했어요. 바람이 아주 빨리 날아가 버렸죠. 파란 차


[1] 옛날 옛날에, 샐리라는 이름의 어린 소녀가 있었습니다. 샐리는 매우 독립적인 아이였죠. 어느 날, 그녀는 큰 성을 발견했어요. 그 용은 어디서 가고 있는지 보고 싶어 했습니다. 그래서 그는 자신의 성으로 달려가 예쁜 성과 용을 가지고 놀았어요. 하지만 그때 갑자기 작은 왕이 나타났습니다! "안녕!"이라고 말했습니다. 나는 여왕이야! 뭐야?"라고 대답했습니다. 왕관이 동의했고, 둘은 주변을 뛰어다녔습니다. 그들은 종일 함께 왕관을 나누었습니다. 곧, 모두 정말 즐거운 시간을 보냈습니다. 샐리와 왕관은 정원에서 놀았고 더 재미있게 놀며


[2] 옛날 옛날에, 루시라는 작은 소녀가 있었어요. 그녀는 매우 영리했어요. 그는 매일 춤추고 음악을 연주하는 것을 좋아했답니다 맑은 어느 날, 루시는 새로운 음악을 연주하고 싶어졌어요. 그래서 그녀는 친구들과 음악을 연주하며 음악을 듣기 위해 연주를 연습했지요. 루시와 그녀의 친구들도 함께 연주할 수 있는 재미있는 드럼을 연주하기 시작했답니다! 친구들은 모두 박수를 치며 환호하며 그럼으로 연주해 주었지요. 그들은 정말 아름다운 음악을 연주했답니다. 모두가 웃으며 트럼펫이 연주했고, 곧 다시 음악을 연주하기 시작했어요. 모든 사람들이 자

=== unconditional ===


[0]  알게 되었답니다!옛날 옛적에 팀이라는 작은 소년이 있었어요. 그는 자신의 장난감 자동차를 가지고 노는 것을 좋아했죠. 어느 날, 팀은 바닥에 놓고 있는 큰 상자를 발견했어요. 그 상자는 매우 궁금해졌죠. 팀은 그 자동차로 놀고 싶어 했죠. 그래서 팀은 상자 안에 뭐가 있는지 알아보고 싶어 했어요. 그들은 장난감 자동차에 가서 장난감들을 올렸어요. 그런데 뜻밖의 일이 벌어졌어요. 장난감 자동차가 움직이기 시작한 거예요! 팀과 샘은 놀랐지만, 하지만 엄마의 말을 듣지 않았어요. 팀이 장난감 자동차가 움직이고 있다는 걸 몰랐거든요. 그


[1]  정말 좋은 일이야."옛날 옛적에 팀이라는 어린 소년이 있었어요. 그는 장난감 장난감을 가지고 노는 것을 좋아했지요. 어느 날, 팀은 방에서 놀 수 있는 큰 상자를 발견했어요. 그 상자에는 장난감이 들어있었답니다! 팀은 매우 신이 나서 엄마에게 물었어요. "엄마, 이 장난감 자동차 좀 가질 수 있을까요?" 팀의 엄마는 "그래, 하지만 조심해야 해. 그런데 먼저 다시 가져갈 수도 있어!"라고 말씀하셨어요. 그래서 그들은 장난감 자동차를 들고 가게로 갔어요. 마침내 팀과 그의 엄마가 장난감 로봇을 보고 도와주려고 했어요. 둘은 함께 자


### 8. 진단 — 고정-t(0.15) acc + infill

In [10]:
g=torch.Generator().manual_seed(0)
def fixed_t_acc(tv_=0.15,n=128):
    cor=tot=0
    for ex in lm_val.select(range(min(n,len(lm_val)))):
        ids=torch.tensor(ex["input_ids"]); m=torch.rand(len(ids),generator=g)<tv_
        if not m.any(): m[0]=True
        inp=ids.clone(); inp[m]=tokenizer.mask_token_id
        with torch.no_grad(): pr=model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
        cor+=(pr[m]==ids[m]).sum().item(); tot+=int(m.sum())
    return cor/tot
print(f"[diag] fixed-t(0.15) top-1 acc = {fixed_t_acc():.3f}   (naive diffusion 0.081)")

[diag] fixed-t(0.15) top-1 acc = 0.652   (naive diffusion 0.081)


## 🛠️ 결과 — coherent 생성과 정확도 0.652

방금 학습한 모델은 `train_loss` 4.13, 고정-$t$(0.15) top-1 정확도 0.652에 도달했습니다. baseline(아무 문맥 없이 균등하게 찍을 때의 손실 $\ln(4000) = 8.29$)에서 한참 내려온 값이고, 모델이 한국어 문맥을 실제로 읽기 시작했다는 신호입니다. 영어 Ch 33의 0.717과 비교하면 교착어인 한국어가 조금 더 까다롭지만, 같은 레시피가 같은 방식으로 작동합니다.

| 구성 | 마스킹 | train_loss | 고정-t(0.15) top-1 acc |
|---|---|---|---|
| **수정판 (Colab T4 30000 step)** | **80/10/10** | **4.13** | **0.652** |
| 수정판 (로컬 8000 step) | 80/10/10 | - | 0.467 |
| (대조) 표준 MLM 15% | 80/10/10 | - | 0.469 |
| (참고) 영어 Ch 33 | 100% [MASK] | - | 0.717 |
| (순진한 방식) diffusion | 100% [MASK] | 7.06 | 0.081 |

로컬 8000 step만으로도 acc가 0.467까지 올라 표준 MLM(0.469)과 사실상 같은 자리에 도달했고, 30000 step을 채우면 0.652까지 갑니다. 표 맨 아래 순진한 100% [MASK] 방식(7.06, 0.081)은 콜레이터에서 80/10/10을 끄면 나오는 결과인데, 왜 그렇게까지 무너지는지는 다음 🔬 해부에서 따져봅니다.

## 생성 결과 — coherent 한국어

진단 acc만 올라간 게 아니라, carry-over 샘플러로 뽑은 문장 자체가 인물과 배경, 대화, 서사를 갖춘 이야기로 바뀝니다. 프롬프트 "옛날 옛날에"로 조건부 생성하거나, 전부 [MASK]인 빈 시퀀스에서 무조건 생성을 돌려도 다음처럼 이어집니다.

> 옛날 옛적에 팀이라는 작은 소년이 있었어요. 팀은 장난감 가지고 노는 것을 매우 좋아했지요. 어느 날, 그는 바닥에서 가장 좋아하는 장난감 자동차를 발견했어요. ... 엄마가 방에 들어오며 '팀아, 내가 네 차를 고쳐줄게!'라고 말했죠. ...

순진한 버전이 내놓던 ",. 함께 하루, 큰.!." 같은 구두점과 고빈도 파편의 나열과 비교하면 차이가 분명합니다. 등장인물(팀), 배경(마을과 공원), 대화, 사건의 흐름이 모두 살아 있습니다. 4.29M짜리 작은 양방향 모델이, 한 줄짜리 콜레이터 수정만으로 도달한 결과입니다.

## 🔬 해부 — 80/10/10이 왜 결정적인가

방금 실습에서 본 coherent한 한국어 생성과 고정-$t$ 정확도 0.652는 80/10/10 마스킹 덕분입니다. 그게 왜 결정적인지를, 순진한 100% `[MASK]` 방식과 직접 대조해 규명합니다.

대조의 흐름은 이렇습니다. 먼저 100% `[MASK]`로 학습했을 때 붕괴의 증상을 숫자로 확정하고(생성이 `",. 함께 하루, 큰.!."` 파편으로 무너지고 `train_loss`가 유니그램 벽에 붙는 모습), 다음으로 데이터·용량·학습량을 늘려 붕괴를 풀어보려는 시도가 모두 막다른 길임을 보입니다. 마지막으로 질문을 뒤집어, 같은 데이터로 자기회귀(Ch 26)와 표준 MLM(80/10/10)은 멀쩡히 학습된다는 대조군을 찾아 범인이 100% `[MASK]` 하나임을 좁혀냅니다.

### 진단 도구 — 고정-t(0.15) top-1 accuracy

생성이 망가질 때 원인은 둘로 갈립니다. 모델이 조건부 구조 $p(x_i \mid x_{\setminus \text{mask}})$를 못 배웠거나, 모델은 멀쩡한데 여러 step을 누적하는 샘플러가 망쳤거나. 생성 결과만 들여다보면 이 둘이 뒤엉켜 책임을 가릴 수 없습니다.

고정-t accuracy는 샘플러를 완전히 배제하고 모델만 떼어내 봅니다. 검증 문장에서 토큰의 15%만($t = 0.15$ 고정) 가린 뒤, 그 자리에서 모델이 내놓는 top-1 예측이 원본과 일치하는 비율을 셉니다. 마스크 비율을 1에서 0으로 내려가며 수십 번 forward하는 생성과 달리 단 한 번의 forward로 끝나므로, 디코딩 노이즈가 끼어들 여지가 없습니다. 순수하게 "양방향 문맥이 주어졌을 때 가린 자리를 맞히는 능력"만 측정합니다.

기준선이 둘 있습니다. 모델이 유니그램 marginal만 외운 채 붕괴하면 이 정확도는 0에 가깝고(가린 자리마다 문맥과 무관하게 고빈도 토큰만 찍을 테니), 반대로 조건부를 제대로 배우면 0.5 안팎까지 올라갑니다. 이 한 숫자로 "모델이 죽었는가 살았는가"를 가릅니다.

### 증상 확정 — loss가 유니그램 벽에 붙어 있다

순진한 이식의 train_loss는 7.06에서 멈췄습니다. 이 숫자가 왜 결정적인지는 기준선과 나란히 놓아야 보입니다. vocab 4000을 아무 문맥 없이 균등하게 찍을 때의 loss가 $\ln(4000) = 8.29$입니다. 7.06은 거기서 겨우 1.2 내려온 값이고, 이 1.2는 정확히 한국어 토큰의 유니그램 엔트로피, 그러니까 "어떤 토큰이 그냥 자주 등장하는가"라는 빈도 정보만큼입니다. 모델은 문맥을 보는 법을 배운 게 아니라 빈도표를 외운 것입니다.

같은 이야기를 다른 각도에서 확인하는 숫자가 고정-t(0.15) top-1 accuracy 0.081입니다. 가린 자리 100개 중 8개만 맞혔다는 뜻이고, 이는 문맥을 거의 활용하지 못한다는 직접 증거입니다. loss 곡선도 같은 그림을 그립니다. 처음 약 250 step 만에 7.0 근처로 내려간 뒤로는 30000 step 내내 평탄하게 누워 있습니다. 유니그램이라는 손쉬운 바닥(attractor)에 빨려 들어간 다음, 거기서 한 발짝도 못 나가는 전형적인 underfitting입니다.

| 측정 항목 | 값 | 의미 |
|---|---|---|
| baseline (균등 추측) | 8.29 | $\ln(4000)$, 문맥·빈도 모두 없음 |
| train_loss (순진한 이식) | 7.06 | 유니그램 엔트로피 근처, 빈도만 학습 |
| 고정-t(0.15) top-1 acc | 0.081 | 가린 자리 100개 중 8개만 복원 |

### 막다른 길 — "왜 안 되나"를 늘려서 풀려 한 시도들

처음 든 의심은 자연스럽게 "덜 학습했나, 모델이 작나, 마스킹이 과한가"였습니다. 그래서 흔히 떠올릴 손잡이를 하나씩 돌려봤습니다. step을 2.67배 늘리고(BLOCK64로 64000 step), 마스크 비율을 $t = \text{rand}^2$로 낮은 쪽에 몰아 쉬운 문제를 더 주고, 모델을 3배(hidden 384/6L, 12.4M)로 키우고, 마스킹률에 $t \le 0.30$ 하드컷을 걸어봤습니다.

결과는 한결같았습니다. 어느 손잡이를 돌려도 loss는 7.0 언저리, accuracy는 0.07-0.08에 그대로 머물렀습니다.

| 시도 | 가설 | train_loss | 고정-t acc | 판정 |
|---|---|---|---|---|
| 순진한 이식 (기준) | — | 7.06 | 0.081 | 붕괴 |
| step 2.67배 (BLOCK64, 64000) | 학습이 부족하다 | 7.08 | 0.068 | 무력 |
| $t = \text{rand}^2$ 저마스킹 편향 | 문제가 너무 어렵다 | 7.19 | 0.068 | 무력 |
| 모델 3배 (12.4M) | 용량이 부족하다 | 7.05 | 0.084 | 무력 |
| 마스킹률 하드컷 $t \le 0.30$ | 마스크가 과하다 | 7.06 | 0.084 | 무력 |

네 시도가 모두 같은 벽에 부딪혔다는 사실 자체가 중요한 정보입니다. loss 곡선이 처음부터 평탄하게 누운 채 움직이지 않는다는 것은, 모델이 유니그램 attractor에 한번 빠지면 step·용량·난이도를 아무리 조절해도 그 바닥을 벗어나지 못한다는 뜻입니다. 바꿔 말하면 범인은 "양"이 아닙니다. 더 오래, 더 크게, 더 쉽게 푸는 방향으로는 문이 열리지 않았습니다.

### 질문 뒤집기 — 안 되는 걸 디버깅할 땐 되는 대조군을 먼저 찾는다

손잡이를 다 돌려보고도 막혔을 때는 질문을 바꿀 차례입니다. "왜 안 되나"를 붙잡고 늘어지는 대신, "그럼 같은 데이터로 무엇은 되는가"를 묻습니다. 되는 사례를 옆에 세워 두면, 안 되는 사례와의 차이가 곧 범인 후보로 좁혀지기 때문입니다.

같은 한국어 TinyStories 데이터, 같은 vocab 4000 토크나이저를 두고 세 가지를 나란히 돌려봤습니다.

| 접근 | 마스킹 / 감독 방식 | 고정-t acc | 생성 결과 |
|---|---|---|---|
| 자기회귀 (Ch 26 AR) | 매 위치 좌→우 순차 감독 | — | coherent ("옛날 옛적에 팀이라는 작은 소년이…") |
| 표준 MLM (고정 15% + 80/10/10) | 가린 자리에 80/10/10 적용 | **0.469** | 정상 학습 (생성 미측정) |
| 순진한 diffusion | 가변 $t$ + 100% [MASK] | 0.081 | 붕괴 ("`,. 함께 하루, 큰.!.`") |

세 줄을 한자리에 놓으니 그림이 분명해집니다. Ch 26의 자기회귀는 같은 데이터로 멀쩡한 이야기를 써냈고, 표준 MLM도 고정-t accuracy 0.469로 정상 학습에 성공했습니다. 데이터가 빈약하거나, 모델이 작거나, 토크나이저가 망가졌다면 이 둘도 함께 무너졌어야 합니다. 그런데 둘 다 잘 됐습니다. 오직 순진한 diffusion만 0.084로 붕괴했습니다.

그렇다면 범인은 데이터도, 용량도, 토크나이저도 아닙니다. 표준 MLM과 순진한 diffusion이 갈라지는 단 하나의 지점, 바로 가린 자리를 채우는 방식에 있습니다. 표준 MLM은 가린 자리에 80/10/10을 적용했고, 순진한 diffusion은 100% [MASK]로 채웠습니다. 이 한 가지 차이가 0.469와 0.084를 갈랐습니다. 다음 절에서 80/10/10이 정확히 무엇을 막아주는지, 그리고 그것을 diffusion에 옮겨 심으면 어떻게 벽이 뚫리는지 살펴봅니다.

## 🚀 삽질 코너 — "조용한 붕괴" 재현하기

80/10/10이 정말 효과의 핵심인지 확인하려면, 그 한 가지만 되돌려 보면 됩니다. 마스킹 대상 자리를 다시 100% `[MASK]`로 덮으면(Ch 33의 영어 레시피 그대로), 한국어에서는 유니그램 붕괴가 그대로 돌아옵니다.

```python
# 80/10/10 분기를 지우고 마스킹 대상을 전부 [MASK]로
mask_sel = torch.rand_like(input_ids.float()) < t   # 마스킹 대상 (불리언)
input_ids[mask_sel] = mask_id                       # 100% [MASK] — 80/10/10 분기 없음
# labels는 그대로 masked_idx에 정답 id, 나머지 -100
```

다른 건 하나도 건드리지 않고 이 분기만 되돌려도, train_loss가 4.13이 아니라 7.06 근처(어휘 4000의 균등 추측 손실 $\ln 4000 \approx 8.29$ 바로 아래)에서 멈춥니다. 고정-$t$(0.15) top-1 정확도는 0.652이 아니라 0.081로 주저앉고, 생성은 ",. 함께 하루, 큰.!."처럼 구두점과 고빈도 파편만 늘어놓습니다. loss 곡선은 처음 약 250 step 만에 7.0으로 평탄해진 뒤 영영 움직이지 않습니다. 유니그램만 배우고 정지한 underfitting입니다.

이 붕괴가 "조용한" 이유는 에러가 전혀 안 나기 때문입니다. 코드는 멀쩡히 돌고 loss도 한 번은 내려가는 듯 보이지만, 모델은 빈도표만 외운 채 멈춰 있습니다. 그래서 loss 절대값을 유니그램 엔트로피와 비교하고, 고정-$t$ top-1 정확도와 실제 생성 텍스트를 함께 봐야 이 조용한 실패를 잡아낼 수 있습니다. 같은 데이터로 AR(Ch 26)과 표준 MLM이 멀쩡히 학습된다는 대조군이 옆에 있으면, 범인이 마스킹 방식 하나로 좁혀집니다.

## 🆚 자기회귀(Ch 26)와의 관계 — 왜 diffusion만 트릭이 필요했나

같은 한국어 TinyStories 데이터로 Ch 26의 자기회귀(AR) 모델은 처음부터 별다른 트릭 없이 coherent한 생성에 성공했습니다("옛날 옛적에 팀이라는 작은 소년이 있었어요…"). 그런데 diffusion만 순진하게 이식하면 무너졌습니다. 차이는 감독 신호의 밀도와 방향에 있습니다.

| | AR (Ch 26) | diffusion (Ch 34) |
|---|---|---|
| 감독 위치 | 매 step 모든 위치 | 마스킹된 일부 자리만 |
| 방향 | 좌→우 순차 | 양방향 |
| 유니그램 지름길 내성 | 높음 (다음 토큰을 늘 문맥으로) | 낮음 (희박한 감독이라 새기 쉬움) |
| 추가 정규화 필요성 | 거의 없음 | 80/10/10이 절실 |

AR은 모든 위치를 매 step 감독하고 좌에서 우로 순차적으로 풀어가므로, 다음 토큰을 맞히려면 어쩔 수 없이 앞 문맥을 봐야 합니다. 유니그램만 외우는 지름길이 통하지 않는 구조입니다. 반면 diffusion은 양방향이면서 한 번에 일부 자리만 감독합니다. 감독이 희박한 만큼 "마스크 자리엔 그냥 흔한 토큰을 찍자"는 지름길로 빠지기 쉽고, 그래서 80/10/10 같은 정규화가 더 절실합니다. 10% 원본 유지는 마스크되지 않은 자리도 문맥으로 검증하게 만들고, 10% 랜덤 교체는 틀린 입력을 교정하도록 강제해서, 모델이 [MASK] 토큰의 존재 자체에만 반응하는 지름길을 닫아버립니다.

정리하면 이렇습니다. AR이 쉬웠던 건 데이터가 쉬워서가 아니라 학습 신호가 촘촘하고 순차적이었기 때문이고, diffusion이 어려웠던 건 신호가 희박하고 양방향이었기 때문입니다. 80/10/10은 그 희박함을 메우는 BERT 시절의 오래된 트릭이고, 한국어라는 더 까다로운 언어에서 비로소 그 진가가 드러났습니다.

## 📦 등장한 라이브러리 정리

이번 장에서 새로 등장했거나, 익숙한 도구를 한국어와 80/10/10 마스킹에 맞게 다르게 쓴 부분만 추렸습니다.

- **80/10/10 diffusion 콜레이터** — 마스킹 대상으로 뽑힌 자리에 무조건 `[MASK]`를 넣던 Ch 33의 콜레이터를, 뽑힌 자리 중 80%만 `[MASK]`로 덮고 10%는 어휘에서 무작위로 고른 토큰으로 바꾸며 10%는 원본 그대로 두는 방식으로 바꿨습니다. 생성용으로 가변 마스킹 비율 $t \sim U(0.05, 1)$은 그대로 유지합니다. loss를 매기는 자리(`labels`)는 80/10/10 분기와 무관하게 "마스킹 대상으로 뽑힌 모든 자리"입니다. 원본을 그대로 둔 10%도 정답을 맞혀야 하는 감독 대상이라는 점이 핵심입니다.

  ```python
  # 마스킹 대상으로 뽑힌 자리(masked_idx)에 80/10/10 분기를 적용
  prob = torch.rand(masked_idx.shape, device=device)
  to_mask   = prob < 0.8                      # 80% → [MASK]
  to_random = (prob >= 0.8) & (prob < 0.9)    # 10% → 무작위 토큰
  # 나머지 10%는 원본 그대로 (입력을 건드리지 않음)
  input_ids[masked_idx[to_mask]]   = mask_id
  input_ids[masked_idx[to_random]] = torch.randint(vocab_size, (to_random.sum(),), device=device)
  # labels는 80/10/10과 무관하게 masked_idx 전체에 정답 id, 나머지는 -100
  ```

- **`BertForMaskedLM` 기본 CE 그대로 쓰기** — Ch 33은 `compute_loss`를 오버라이드해 시간가중 $1/t$를 직접 곱했지만, 이번 장은 `BertForMaskedLM`이 내부에서 계산하는 기본 손실(마스크 자리 평균 교차엔트로피, plain CE)을 그대로 받습니다. `labels`에 마스크 자리 정답 id를, 나머지에 `-100`(`ignore_index`)을 채워 넘기면 모델이 알아서 마스크 자리 평균 CE를 돌려줍니다. 표준 MLM과 손실 형태를 똑같이 맞춰 두 실험을 같은 잣대로 비교하려는 의도입니다.

  ```python
  from transformers import BertConfig, BertForMaskedLM

  config = BertConfig(
      vocab_size=4000, hidden_size=256, num_hidden_layers=4,
      num_attention_heads=4, intermediate_size=1024, max_position_embeddings=128,
  )
  model = BertForMaskedLM(config)        # 약 4.29M
  out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
  loss = out.loss                        # 마스크 자리 평균 CE, 별도 가중 없음
  ```

- **한국어 ByteLevel BPE (vocab 4000)** — `tokenizers`로 `g0ster/TinyStories-Korean` 코퍼스에 ByteLevel BPE를 직접 학습합니다. `[PAD]`, `[UNK]`, `[MASK]` 특수 토큰을 더하고, `initial_alphabet`에 전체 바이트(256개)를 넣어 어떤 한글 문자라도 `[UNK]`로 빠지지 않게 합니다. 영어 Ch 33은 vocab 2048로 충분했지만, 한국어는 교착어라 조사·어미 조합이 다양해 4000으로 약간 키웠습니다(Ch 26과 같은 크기).

  ```python
  from tokenizers import ByteLevelBPETokenizer

  tok = ByteLevelBPETokenizer()
  tok.train_from_iterator(
      text_iter,                                   # 한국어 동화 텍스트 제너레이터
      vocab_size=4000,
      special_tokens=["[PAD]", "[UNK]", "[MASK]"],
      initial_alphabet=[bytes([b]).decode("latin-1") for b in range(256)],
  )
  ```

- **`<|endoftext|>` 경계로 story 복원** — `g0ster/TinyStories-Korean`은 여러 이야기가 `<|endoftext|>`로 이어진 한 덩어리입니다. 이 경계로 잘라 50,000개 story를 복원한 뒤 토큰화·청킹해 75,941개 chunk(약 9.7M 토큰)를 만듭니다. Ch 26과 같은 데이터·같은 전처리라 "언어 축"의 변화만 깨끗하게 드러납니다.

## 🎯 체크포인트 질문

1. Ch 33의 레시피(100% `[MASK]` + 시간가중 $1/t$)를 한국어로 그대로 옮겼더니 train_loss가 7.06에서 멈췄습니다. 한국어 어휘 4000개의 균등 추측의 손실 상한이 $\ln 4000 \approx 8.29$라는 점과 묶어, 이 7.06이라는 숫자 하나로 "모델이 조건부 문맥이 아니라 단어 빈도만 외웠다"고 진단할 수 있는 이유를 설명해 보세요.

2. step을 2.67배(64000)로 늘려도, 모델을 3배(12.4M)로 키워도, 마스킹 비율을 저쪽으로 편향시켜도 loss는 전부 7.0 근처에 고착됐습니다. 이 네 가지 반증 실험이 함께 가리키는 결론은 무엇이며, 그래서 범인이 "학습량·용량·데이터"가 아니라 "마스킹 방식"이라고 좁힐 수 있는 까닭은 무엇일까요?

3. 80/10/10에서 10%를 원본 그대로 두는 분기와 10%를 무작위 토큰으로 바꾸는 분기는 각각 모델에게 어떤 압박을 줍니까? 이 두 분기가 "`[MASK]`를 보면 그냥 고빈도 토큰을 찍는" 지름길을 어떻게 막는지 말해 보세요.

4. 같은 한국어 데이터로 AR(Ch 26)은 coherent 생성에 성공했고 표준 MLM도 고정-$t$ 정확도 0.469를 냈는데, 순진한 diffusion만 0.084로 무너졌습니다. 이 대조군 세 개를 나란히 놓는 것이 "데이터·토크나이저가 아니라 마스킹이 범인"이라는 진단에 왜 결정적인지 설명해 보세요.

## ❓ FAQ

**Q1. 영어 Ch 33은 100% `[MASK]`로도 잘 됐는데, 왜 한국어는 똑같이 했더니 무너지나요?**

언어의 통계적 성질이 다르기 때문입니다. 한국어는 교착어라 조사("-는", "-을", "-에")와 어미("-어요", "-습니다")가 위치만으로도 상당히 예측됩니다. 즉 문맥을 깊이 읽지 않고 유니그램 빈도만 따라가도 그럴듯하게 찍히는 자리가 많습니다. 입력 자리가 전부 `[MASK]` 하나로만 덮이면, 모델 입장에서는 "`[MASK]`를 보면 그 자리에 가장 흔한 토큰을 찍자"는 지름길이 너무 매력적입니다. 그래서 첫 250 step 안에 유니그램만 빠르게 익히고 그 자리에서 멈춥니다(train_loss 7.06, 유니그램 엔트로피 근처). 반면 영어 TinyStories는 어휘와 구조가 단순·반복적이라 같은 지름길의 유혹이 약해 100% `[MASK]`로도 버틴 것입니다. 데이터가 바뀌면 같은 레시피의 안전 마진도 바뀝니다.

**Q2. 80/10/10에서 10% 원본 유지와 10% 무작위 교체는 각각 무슨 역할을 하나요?**

서로 다른 지름길 두 개를 막습니다. **10% 원본 유지**는 "마스크 안 된 자리도 그냥 믿지 말고 문맥으로 검증하라"는 압박입니다. 입력에 `[MASK]`가 없는 멀쩡한 토큰이 있어도 그 자리가 정답 감독 대상일 수 있으니, 모델은 "`[MASK]`인 자리만 추론하면 된다"는 위치 기반 지름길을 못 씁니다. **10% 무작위 교체**는 "틀린 입력을 교정하라"는 압박입니다. 문맥상 말이 안 되는 토큰이 끼어 있어도 정답을 복원해야 하니, 입력 토큰을 액면 그대로 베끼는 지름길이 막힙니다. 두 분기가 함께 "`[MASK]` → 고빈도 토큰 찍기"라는 단일 지름길을 봉쇄해, 모델이 어쩔 수 없이 문맥을 읽게 만듭니다.

```python
prob = torch.rand(masked_idx.shape)
to_mask   = prob < 0.8                      # 80% → [MASK]
to_random = (prob >= 0.8) & (prob < 0.9)    # 10% → 무작위 토큰 (교정 압박)
# 나머지 10% → 원본 유지 (검증 압박). labels에는 셋 다 정답 id가 들어감
```

**Q3. Ch 33의 시간가중 $1/t$ 손실을 왜 이번엔 plain CE로 바꿨나요? 성질이 달라지는 것 아닌가요?**

기댓값 위에서 둘은 본질적으로 같기 때문입니다. $1/t$ 가중은 "마스킹 비율이 작은 step일수록 그 적은 마스크 자리를 크게 친다"는 보정인데, 한 자리당 평균으로 환산하면 마스크 토큰당 평균 CE와 상수배 관계입니다. 즉 기댓값상 마스크 토큰당 평균 CE(plain CE가 계산하는 그 값)와 같은 목적함수를 가리킵니다. 이번 장에서 plain CE를 택한 진짜 이유는 **표준 MLM 대조군과 손실 형태를 정확히 맞추기 위해서**입니다. 표준 MLM(고정 15% + 80/10/10 + plain CE)이 같은 한국어 데이터로 0.469를 내는데, diffusion 쪽도 같은 plain CE를 쓰면 "달라진 건 마스킹 비율을 가변으로 둔 것뿐"이 되어 두 결과를 깨끗하게 비교할 수 있습니다.

```python
out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
loss = out.loss     # BertForMaskedLM 기본 = 마스크 자리 평균 CE (plain CE)
```

**Q4. vocab을 왜 영어(2048)보다 큰 4000으로 키웠나요?**

한국어가 교착어라서입니다. 영어는 단어 사이가 공백으로 끊겨 BPE가 비교적 적은 조각으로도 어휘를 덮지만, 한국어는 어간에 조사·어미가 붙어 한 어절이 여러 형태로 변주됩니다("소년이", "소년은", "소년에게"). vocab이 너무 작으면 이런 변주가 잘게 쪼개져 한 이야기를 표현하는 데 토큰이 더 많이 들고 시퀀스가 길어집니다. 4000으로 키우면 자주 쓰는 조사·어미 조합이 적당한 단위로 묶여 시퀀스가 짧아지고 학습이 안정됩니다. Ch 26의 한국어 AR도 같은 이유로 4000을 썼고, 같은 크기로 맞춰 두면 AR과 diffusion 비교에서 토크나이저 변수를 제거할 수 있습니다.

**Q5. 고정-$t$ accuracy로 무엇을 알 수 있나요? 왜 생성 품질만 보지 않나요?**

샘플러를 떼어내고 "모델 자체"의 실력을 재기 위해서입니다. 생성 품질은 모델뿐 아니라 carry-over·반복억제 같은 디코딩 설정에도 좌우돼서, 출력만 보면 무엇이 좋아졌는지 분리가 안 됩니다. 고정-$t$ top-1 accuracy는 마스킹 비율을 고정값(여기선 0.15)으로 두고 마스크 자리에서 정답을 1순위로 맞히는 비율만 봅니다. 순진한 diffusion은 0.081로 거의 찍기 수준이었지만, 80/10/10 교정 후 0.652까지 올랐습니다(영어 Ch 33은 0.717). 로컬 8000 step에서는 0.467로 표준 MLM의 0.469와 동급이었습니다. 이 한 숫자만으로 "모델이 문맥 조건부를 배웠는가"를 샘플러와 무관하게 판정할 수 있습니다.

```python
# 마스킹 비율을 0.15로 고정하고 마스크 자리 top-1 일치율만 측정
masked = make_masked_batch(input_ids, ratio=0.15)
pred = model(**masked).logits.argmax(-1)
acc = (pred[mask_pos] == gold[mask_pos]).float().mean()   # 샘플러 무관
```

**Q6. AR(Ch 26)이 같은 데이터로 그냥 잘 되는데, 왜 굳이 diffusion을 쓰나요?**

이 장에서 AR은 "대조군"으로서 결정적입니다. AR이 같은 한국어 데이터·같은 용량으로 coherent하게 생성됐다는 사실이, diffusion의 붕괴가 데이터나 토크나이저나 모델 크기 탓이 아니라 마스킹 방식 탓임을 증명해 주기 때문입니다. AR은 모든 위치를 매 step 감독하고 좌→우로 순차 생성하는 반면, diffusion은 임의 위치를 병렬로 채웁니다. 이 병렬·임의 순서가 주는 잠재적 장점(예: 양방향 문맥 활용, 위치 제약 없는 채우기) 때문에 diffusion 계열을 연구·실험하는 가치가 있고, 이 장은 그 계열을 작은 규모에서 "제대로 학습되게" 만드는 레시피를 확보하는 자리입니다. 실용적 선택이라기보다, AR이라는 기준점 옆에 diffusion을 세워 무엇이 같고 다른지 직접 만져 보는 것이 목적입니다.

**Q7. 교정 후에도 영어(0.717)보다 한국어(0.652)가 낮은데, 더 키워야 하나요?**

이 격차는 붕괴가 아니라 언어 난이도 차이로 보는 게 맞습니다. 한국어는 조사·어미 변주가 많아 같은 규모에서 토큰 예측이 영어보다 까다롭습니다. 중요한 건 절대 수치가 아니라 "벽을 넘었는가"입니다. 순진한 레시피가 0.081에서 멈춰 있던 것을 80/10/10으로 0.652까지 끌어올렸고, train_loss도 7.0의 벽을 넘어 4.13까지 내려갔으며(30000 step, Colab T4, 20.11분), 무조건 생성에서 인물(팀)과 배경(마을·공원), 대화, 서사가 갖춰진 동화가 나옵니다. 자잘한 흠은 모델·데이터를 키우면 줄지만 그건 T4 30분 예산 밖의 이야기입니다. 이 장의 목표선("작은 한국어 diffusion LM이 coherent하게 생성된다")은 이미 넘었습니다.

## 다음 챕터 예고 — Phase 5 마무리

여기까지 오며 작은 mask-diffusion을 영어로 살려 내고(Ch 33), 같은 레시피가 한국어에서 무너지는 지점을 진단해 BERT의 80/10/10 마스킹으로 되살렸습니다(Ch 34). 언어가 바뀌면 멀쩡하던 레시피가 조용히 무너질 수 있다는 것, 그리고 그 붕괴를 loss 절대값·고정-$t$ 정확도·대조군이라는 세 잣대로 잡아내는 법을 함께 익혔습니다.

이로써 diffusion 라인의 핵심 골격은 갖춰졌습니다. 다음으로는 지금까지 손으로 쌓아 온 자기회귀 계열과 diffusion 계열을 한 호흡으로 되돌아보며, 두 패러다임이 학습 신호의 밀도와 생성 방식에서 어떻게 갈라지는지 정리할 차례입니다. 구체적인 다음 주제는 정해지는 대로 안내하겠습니다.